# Multi-task learning: multilingual multilabel NER

This notebook trains one **XLM-RoBERTa** token classification model on multilingual NER data (IOB) using a multi-task setup with **three separate classification heads**, one for each entity type (**disease**, **symptom**, and **procedure**).

**References:** 

[Token classification (Hugging Face)](https://huggingface.co/docs/transformers/tasks/token_classification)

[Multi-Task Model for Fake and Hate Probability Prediction with BERT](https://www.analyticsvidhya.com/blog/2023/06/building-a-multi-task-model-for-fake-and-hate-probability-prediction-with-bert/)

## Dependencies

First of all, we install all the necessary libraries:

In [1]:
pip install huggingface_hub transformers datasets evaluate seqeval nlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 48.0 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=38f826cdf95d21f856903cae1a069511fb618a3db2dfb47fe03a15357e2ca67e
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


## Imports
We import all necessary libraries:

In [2]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import transformers
import nlp
import torch.optim as optim
from torch.optim import AdamW
import logging
logging.basicConfig(level=logging.INFO)
from transformers import AutoModel, AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
from huggingface_hub import PyTorchModelHubMixin
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from keras.utils import pad_sequences
from torch.nn.utils.rnn import pad_sequence

## Hugging Face Hub login

In [3]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Data downloading

Download MultiClinAI Shared Task Training Set for MultiClinNER from [Zenodo](https://zenodo.org/records/19098018).

## Data uploading and formatting

Place the CSVs on Drive (or update paths). Expected layout (**IOB**), one row per token; sentence boundaries use blank lines / NaN tokens:

- **Disease** corpus: `/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/{cz,en,es,nl,sv,ro,it}/disease/kfold_5/{fold_*}/{train,val}/{lang}-disease-{fold}-{split}.csv`

- **Symptom** corpus: `/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/{cz,en,es,nl,sv,ro,it}/symptom/kfold_5/{fold_*}/{train,val}/{lang}-symptom-{fold}-{split}.csv`

- **Procedure** corpus: `/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/{cz,en,es,nl,sv,ro,it}/procedure/kfold_5/{fold_*}/{train,val}/{lang}-procedure-{fold}-{split}.csv`

Then, we connect to google drive, where we have stored our data:

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## CSV paths

Load the csv files. In this set up, each classification head corresponds to an entity type:

In [5]:
# Training data for DISEASE NER task

train_es_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/es/disease/kfold_5/fold_1/train/es-disease-fold_1-train.csv'
train_it_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/it/disease/kfold_5/fold_1/train/it-disease-fold_1-train.csv'
train_ro_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/ro/disease/kfold_5/fold_1/train/ro-disease-fold_1-train.csv'
train_en_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/en/disease/kfold_5/fold_1/train/en-disease-fold_1-train.csv'
train_nl_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/nl/disease/kfold_5/fold_1/train/nl-disease-fold_1-train.csv'
train_sv_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/sv/disease/kfold_5/fold_4/train/sv-disease-fold_4-train.csv'
train_cz_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/cz/disease/kfold_5/fold_1/train/cz-disease-fold_1-train.csv'

dev_es_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/es/disease/kfold_5/fold_1/val/es-disease-fold_1-val.csv'
dev_it_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/it/disease/kfold_5/fold_1/val/it-disease-fold_1-val.csv'
dev_ro_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/ro/disease/kfold_5/fold_1/val/ro-disease-fold_1-val.csv'
dev_en_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/en/disease/kfold_5/fold_1/val/en-disease-fold_1-val.csv'
dev_nl_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/nl/disease/kfold_5/fold_1/val/nl-disease-fold_1-val.csv'
dev_sv_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/sv/disease/kfold_5/fold_4/val/sv-disease-fold_4-val.csv'
dev_cz_disease = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/cz/disease/kfold_5/fold_1/val/cz-disease-fold_1-val.csv'

# Training data for SYMPTOM NER task

train_es_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/es/symptom/kfold_5/fold_1/train/es-symptom-fold_1-train.csv'
train_it_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/it/symptom/kfold_5/fold_1/train/it-symptom-fold_1-train.csv'
train_ro_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/ro/symptom/kfold_5/fold_1/train/ro-symptom-fold_1-train.csv'
train_en_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/en/symptom/kfold_5/fold_1/train/en-symptom-fold_1-train.csv'
train_nl_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/nl/symptom/kfold_5/fold_1/train/nl-symptom-fold_1-train.csv'
train_sv_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/sv/symptom/kfold_5/fold_4/train/sv-symptom-fold_4-train.csv'
train_cz_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/cz/symptom/kfold_5/fold_1/train/cz-symptom-fold_1-train.csv'

dev_es_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/es/symptom/kfold_5/fold_1/val/es-symptom-fold_1-val.csv'
dev_it_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/it/symptom/kfold_5/fold_1/val/it-symptom-fold_1-val.csv'
dev_ro_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/ro/symptom/kfold_5/fold_1/val/ro-symptom-fold_1-val.csv'
dev_en_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/en/symptom/kfold_5/fold_1/val/en-symptom-fold_1-val.csv'
dev_nl_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/nl/symptom/kfold_5/fold_1/val/nl-symptom-fold_1-val.csv'
dev_sv_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/sv/symptom/kfold_5/fold_4/val/sv-symptom-fold_4-val.csv'
dev_cz_symptom = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/cz/symptom/kfold_5/fold_1/val/cz-symptom-fold_1-val.csv'

# Training data for PROCEDURE NER task

train_es_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/es/procedure/kfold_5/fold_1/train/es-procedure-fold_1-train.csv'
train_it_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/it/procedure/kfold_5/fold_1/train/it-procedure-fold_1-train.csv'
train_ro_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/ro/procedure/kfold_5/fold_1/train/ro-procedure-fold_1-train.csv'
train_en_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/en/procedure/kfold_5/fold_1/train/en-procedure-fold_1-train.csv'
train_nl_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/nl/procedure/kfold_5/fold_1/train/nl-procedure-fold_1-train.csv'
train_sv_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/sv/procedure/kfold_5/fold_4/train/sv-procedure-fold_4-train.csv'
train_cz_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/cz/procedure/kfold_5/fold_1/train/cz-procedure-fold_1-train.csv'

dev_es_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/es/procedure/kfold_5/fold_1/val/es-procedure-fold_1-val.csv'
dev_it_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/it/procedure/kfold_5/fold_1/val/it-procedure-fold_1-val.csv'
dev_ro_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/ro/procedure/kfold_5/fold_1/val/ro-procedure-fold_1-val.csv'
dev_en_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/en/procedure/kfold_5/fold_1/val/en-procedure-fold_1-val.csv'
dev_nl_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/nl/procedure/kfold_5/fold_1/val/nl-procedure-fold_1-val.csv'
dev_sv_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/sv/procedure/kfold_5/fold_4/val/sv-procedure-fold_4-val.csv'
dev_cz_procedure = '/content/drive/MyDrive/MultiClinAI_NER2/monolingual_2/monolabel/cz/procedure/kfold_5/fold_1/val/cz-procedure-fold_1-val.csv'



## Load CSVs into DataFrames

Read the csv files to pandas dataframes, specifying the delimiter as ','.

Column layout: `ner`, `start`, `end`, `token`.

In [6]:
pd.options.display.max_colwidth = 900000

column_names = ['ner', 'start', 'end', 'token']

# Training data for DISEASE NER task

train_es_disease_df = pd.read_csv(train_es_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_it_disease_df = pd.read_csv(train_it_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_ro_disease_df = pd.read_csv(train_ro_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_en_disease_df = pd.read_csv(train_en_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_nl_disease_df = pd.read_csv(train_nl_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_sv_disease_df = pd.read_csv(train_sv_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_cz_disease_df = pd.read_csv(train_cz_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')

dev_es_disease_df = pd.read_csv(dev_es_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_it_disease_df = pd.read_csv(dev_it_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_ro_disease_df = pd.read_csv(dev_ro_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_en_disease_df = pd.read_csv(dev_en_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_nl_disease_df = pd.read_csv(dev_nl_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_sv_disease_df = pd.read_csv(dev_sv_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_cz_disease_df = pd.read_csv(dev_cz_disease, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')

# Training data for SYMPTOM NER task

train_es_symptom_df = pd.read_csv(train_es_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_it_symptom_df = pd.read_csv(train_it_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_ro_symptom_df = pd.read_csv(train_ro_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_en_symptom_df = pd.read_csv(train_en_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_nl_symptom_df = pd.read_csv(train_nl_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_sv_symptom_df = pd.read_csv(train_sv_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_cz_symptom_df = pd.read_csv(train_cz_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')

dev_es_symptom_df = pd.read_csv(dev_es_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_it_symptom_df = pd.read_csv(dev_it_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_ro_symptom_df = pd.read_csv(dev_ro_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_en_symptom_df = pd.read_csv(dev_en_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_nl_symptom_df = pd.read_csv(dev_nl_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_sv_symptom_df = pd.read_csv(dev_sv_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_cz_symptom_df = pd.read_csv(dev_cz_symptom, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')

# Training data for PROCEDURE NER task

train_es_procedure_df = pd.read_csv(train_es_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_it_procedure_df = pd.read_csv(train_it_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_ro_procedure_df = pd.read_csv(train_ro_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_en_procedure_df = pd.read_csv(train_en_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_nl_procedure_df = pd.read_csv(train_nl_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_sv_procedure_df = pd.read_csv(train_sv_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
train_cz_procedure_df = pd.read_csv(train_cz_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')

dev_es_procedure_df = pd.read_csv(dev_es_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_it_procedure_df = pd.read_csv(dev_it_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_ro_procedure_df = pd.read_csv(dev_ro_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_en_procedure_df = pd.read_csv(dev_en_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_nl_procedure_df = pd.read_csv(dev_nl_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_sv_procedure_df = pd.read_csv(dev_sv_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')
dev_cz_procedure_df = pd.read_csv(dev_cz_procedure, delimiter = ',', names=column_names, skip_blank_lines=False, encoding='utf-8')


## Merge train + validation and keep `token` / `ner`

Training uses all train+val data. Thus, we merge the train and validation sets corresponding to the same entity type across all languages, adding empty rows to separate the corpora so that sentence grouping restarts at the boundaries.

In [7]:
# Create empty row with same columns
empty_row = pd.DataFrame([[None]*len(train_es_disease_df.columns)], columns=train_es_disease_df.columns)

# Merge with empty row between them

# DISEASE training data

train_disease_df = pd.concat([train_es_disease_df, empty_row, dev_es_disease_df, empty_row,
                              train_it_disease_df, empty_row, dev_it_disease_df, empty_row,
                              train_ro_disease_df, empty_row, dev_ro_disease_df, empty_row,
                              train_en_disease_df, empty_row, dev_en_disease_df, empty_row,
                              train_nl_disease_df, empty_row, dev_nl_disease_df, empty_row,
                              train_sv_disease_df, empty_row, dev_sv_disease_df, empty_row,
                              train_cz_disease_df, empty_row, dev_cz_disease_df], ignore_index=True)

# SYMPTOM training data

train_symptom_df = pd.concat([train_es_symptom_df, empty_row, dev_es_symptom_df, empty_row,
                              train_it_symptom_df, empty_row, dev_it_symptom_df, empty_row,
                              train_ro_symptom_df, empty_row, dev_ro_symptom_df, empty_row,
                              train_en_symptom_df, empty_row, dev_en_symptom_df, empty_row,
                              train_nl_symptom_df, empty_row, dev_nl_symptom_df, empty_row,
                              train_sv_symptom_df, empty_row, dev_sv_symptom_df, empty_row,
                              train_cz_symptom_df, empty_row, dev_cz_symptom_df], ignore_index=True)

# PROCEDURE training data

train_procedure_df = pd.concat([train_es_procedure_df, empty_row, dev_es_procedure_df, empty_row,
                                train_it_procedure_df, empty_row, dev_it_procedure_df, empty_row,
                                train_ro_procedure_df, empty_row, dev_ro_procedure_df, empty_row,
                                train_en_procedure_df, empty_row, dev_en_procedure_df, empty_row,
                                train_nl_procedure_df, empty_row, dev_nl_procedure_df, empty_row,
                                train_sv_procedure_df, empty_row, dev_sv_procedure_df, empty_row,
                                train_cz_procedure_df, empty_row, dev_cz_procedure_df], ignore_index=True)

/tmp/ipykernel_648/2687862787.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  train_disease_df = pd.concat([train_es_disease_df, empty_row, dev_es_disease_df, empty_row,
/tmp/ipykernel_648/2687862787.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  train_symptom_df = pd.concat([train_es_symptom_df, empty_row, dev_es_symptom_df, empty_row,
/tmp/ipykernel_648/2687862787.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future ver

We need to make sure all labels are in English:

In [8]:
label_map = {'B-ENFERMEDAD':'B-DISEASE', 'I-ENFERMEDAD':'I-DISEASE', 'B-SINTOMA':'B-SYMPTOM', 'I-SINTOMA':'I-SYMPTOM', 'B-PROCEDIMIENTO':'B-PROCEDURE', 'I-PROCEDIMIENTO':'I-PROCEDURE'}

train_disease_df['ner'] = train_disease_df['ner'].replace(label_map)
train_symptom_df['ner'] = train_symptom_df['ner'].replace(label_map)
train_procedure_df['ner'] = train_procedure_df['ner'].replace(label_map)

We need to modify the dataframes columns in order to have the correct data for MTL NER model training. We need to delete the 'start' and 'end' columns in the dataframes and change column position ('token' first, 'ner' after). We only keep `token` and `ner` columns (in that order).

In [9]:
# Define a function to create a new DataFrame with only 'token' and 'ner' columns in the correct order
def create_new_dataframe(original_df):
    new_df = original_df[['token', 'ner']].copy()
    return new_df

# Create new dfs
train_disease_df = create_new_dataframe(train_disease_df)
train_symptom_df = create_new_dataframe(train_symptom_df)
train_procedure_df = create_new_dataframe(train_procedure_df)

## Sentence-level table (`id`, `tokens`, `ner`)

We need to have a sentence for each id (not a single token). For this, we modify the dataframes:

In [10]:
def transform_df(old_df):
    # Initialize lists to store modified data
    ids = []
    tokens = []
    ners = []

    # Initialize variables for the current sentence
    current_id = 0
    current_tokens = []
    current_ners = []

    # Iterate through the train DataFrame
    for index, row in old_df.iterrows():
        token = row['token']
        ner = row['ner']

        if pd.notna(token):
            current_tokens.append(str(token))
            current_ners.append(str(ner))
        else:  # Sentence boundary detected (NaN)
            if current_tokens:
                ids.append(current_id)
                tokens.append(current_tokens)
                ners.append(current_ners)

            # Reset for the next sentence
            current_id += 1
            current_tokens = []
            current_ners = []

    # Handle the last sentence if the DataFrame doesn't end with a NaN row
    if current_tokens:
        ids.append(current_id)
        tokens.append(current_tokens)
        ners.append(current_ners)

    # Create a new DataFrame with the desired structure
    new = {
        'id': ids,
        'tokens': tokens,
        'ner': ners
    }

    new_df = pd.DataFrame(new)
    return new_df

We apply the function to the dataframes:

In [11]:
train_disease_df = transform_df(train_disease_df)
train_symptom_df = transform_df(train_symptom_df)
train_procedure_df = transform_df(train_procedure_df)

## Hugging Face `DatasetDict` and label maps

We now convert the dataframe to a **HuggingFace DatasetDict**.

`id2label` / `label2id` use **sorted** unique labels so IDs are stable across runs.

We convert the final pandas dataframes (dfs) into a single HuggingFace DatasetDict, where the **tasks** are the top-level keys ("disease", "symptom", "procedure") and each one maps to its own dataset with a "train" split:

In [12]:
dfs = {
    "disease": train_disease_df,
    "symptom": train_symptom_df,
    "procedure": train_procedure_df,
}

datasets_dict = {
    task: DatasetDict({"train": Dataset.from_pandas(df)})
    for task, df in dfs.items()
}

Now we proceed with the implementation of the necessary feature engineering. The first step is to build a dictionary (label2id) that assigns a unique integer value to every ner label from the corpus. We also construct a reversed dictionary that maps indices to ner labels (id2label).

In [13]:
label_maps = {}

for task, datasetdict in datasets_dict.items():
    dataset = datasetdict["train"]

    unique_labels = set()
    for labels in dataset["ner"]:
        unique_labels.update(labels)

    label_list = sorted(unique_labels)

    id2label = {i: label for i, label in enumerate(label_list)}
    label2id = {label: i for i, label in enumerate(label_list)}

    label_maps[task] = {
        "id2label": id2label,
        "label2id": label2id
    }

## Tokenizer and label alignment
Now we need to convert all NER values to the corresponding IDs:

In [14]:
# Define a conversion function
def encode_batch(batch, label2id):
    batch["ner"] = [
        [label2id[label] for label in seq]
        for seq in batch["ner"]
    ]
    return batch

In [15]:
# Apply the conversion funtion to each dataset (task) in the DatasetDict (datasets_dict)
encoded_datasets_dict = {}

for task, datasetdict in datasets_dict.items():
    label2id = label_maps[task]["label2id"]

    encoded_datasets_dict[task] = datasetdict.map(
        lambda batch: encode_batch(batch, label2id),
        batched=True
    )

Map:   0%|          | 0/332753 [00:00<?, ? examples/s]

Map:   0%|          | 0/331944 [00:00<?, ? examples/s]

Map:   0%|          | 0/332579 [00:00<?, ? examples/s]

We create our tokenizer object (`model_checkpoint` can be changed) and a function to align the labels to the tokens:

In [16]:
# Define model_checkpoint
model_checkpoint = 'FacebookAI/xlm-roberta-base'

#Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Function to tokenize and align labels for each dataset
def tokenize_and_align_labels(examples, max_length=512):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=max_length)

    labels = []
    for i, label in enumerate(examples[f"ner"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = {task: {} for task in encoded_datasets_dict.keys()}

for task, dataset in encoded_datasets_dict.items():
    tokenized_datasets[task]["train"] = dataset["train"].map(
        tokenize_and_align_labels,
        batched=True
    )

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/332753 [00:00<?, ? examples/s]

Map:   0%|          | 0/331944 [00:00<?, ? examples/s]

Map:   0%|          | 0/332579 [00:00<?, ? examples/s]

Now we need to create DataLoader for each task:

In [17]:
def create_dataset(tokenized_dataset):
    # Convert lists to torch tensors
    input_ids = [torch.tensor(seq) for seq in tokenized_dataset['input_ids']]
    attention_masks = [torch.tensor(seq) for seq in tokenized_dataset['attention_mask']]
    labels = [torch.tensor(seq) for seq in tokenized_dataset['labels']]

    # Pad sequences
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks = pad_sequence(attention_masks, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)  # Use -100 to ignore padding in the loss

    return TensorDataset(input_ids, attention_masks, labels)


train_dataloaders = {}

for task in tokenized_datasets.keys():
    train_data = create_dataset(tokenized_datasets[task]['train'])

    train_sampler = RandomSampler(train_data)

    train_dataloaders[task] = DataLoader(train_data, sampler=train_sampler, batch_size=32, num_workers=2, pin_memory=True)

## Define the Multi-task model architecture for NER

In [19]:
class MultiTaskNERModel(nn.Module):
    def __init__(self, model_checkpoint, num_labels):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_checkpoint)
        self.dropout = nn.Dropout(0.1)

        hidden_size = self.encoder.config.hidden_size

        self.classifiers = nn.ModuleDict({
            task: nn.Linear(hidden_size, n_labels)
            for task, n_labels in num_labels.items()
        })

    def forward(self, input_ids, attention_mask, task):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = self.dropout(outputs.last_hidden_state)

        logits = self.classifiers[task](sequence_output)

        return logits

## Initialize the model

In [20]:
num_labels = {
    task: len(label_maps[task]["label2id"])
    for task in label_maps
}

model = MultiTaskNERModel(
    model_checkpoint=model_checkpoint,
    num_labels=num_labels
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Train the model

In [21]:
use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

for epoch in range(10):
    model.train()
    for task in train_dataloaders.keys():
        for step, batch in enumerate(train_dataloaders[task]):
            input_ids = batch[0].to(device, non_blocking=True)
            attention_mask = batch[1].to(device, non_blocking=True)
            labels = batch[2].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(input_ids, attention_mask, task)
                loss = criterion(
                    logits.view(-1, logits.shape[-1]),
                    labels.view(-1)
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            if step % 50 == 0:
                print(f"Epoch {epoch} | Task {task} | Step {step} | Loss {loss.item():.4f}")

/tmp/ipykernel_648/2379005134.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_648/2379005134.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Streaming output truncated to the last 5000 lines.
Epoch 1 | Task procedure | Step 10000 | Loss 0.0748
Epoch 1 | Task procedure | Step 10050 | Loss 0.0644
Epoch 1 | Task procedure | Step 10100 | Loss 0.0454
Epoch 1 | Task procedure | Step 10150 | Loss 0.0972
Epoch 1 | Task procedure | Step 10200 | Loss 0.0539
Epoch 1 | Task procedure | Step 10250 | Loss 0.0476
Epoch 1 | Task procedure | Step 10300 | Loss 0.0776
Epoch 1 | Task procedure | Step 10350 | Loss 0.0385
Epoch 2 | Task disease | Step 0 | Loss 0.3434
Epoch 2 | Task disease | Step 50 | Loss 0.1319
Epoch 2 | Task disease | Step 100 | Loss 0.0235
Epoch 2 | Task disease | Step 150 | Loss 0.0780
Epoch 2 | Task disease | Step 200 | Loss 0.0892
Epoch 2 | Task disease | Step 250 | Loss 0.0477
Epoch 2 | Task disease | Step 300 | Loss 0.0310
Epoch 2 | Task disease | Step 350 | Loss 0.0509
Epoch 2 | Task disease | Step 400 | Loss 0.0780
Epoch 2 | Task disease | Step 450 | Loss 0.0951
Epoch 2 | Task disease | Step 500 | Loss 0.0413
Epoch 2 

## Save the model

In [22]:
# Define the path where you want to save the model
save_path = "/content/drive/MyDrive/MultiClinAI_NER2"

# Make sure the directory exists, if not create it
os.makedirs(save_path, exist_ok=True)

# Save the model's state dictionary after the training loop
model_save_path = os.path.join(save_path, "MultiClinAI_XLM-R_Multilingual_Multilabel.pt")

torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to /content/drive/MyDrive/MultiClinAI_NER2/MultiClinAI_XLM-R_Multilingual_Multilabel.pt
